# SpliceTransformer Splice Site Prediction

![SpliceTransformer Splice Site Prediction](https://proto-bio.github.io/proto-assets/images/tool/splice_transformer/hero.png)

This notebook demonstrates `run_splice_transformer`, which predicts tissue-specific splice site locations from raw DNA sequence. Given a target sequence flanked by genomic context, the model returns per-position probabilities for three splice categories (neither, acceptor, donor) along with 15 tissue-specific usage channels, making it useful for variant interpretation, synthetic intron design, and analysis of tissue-specific alternative splicing.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("splice_transformer")
display_overview("splice_transformer")
display_docs_section("splice_transformer", "Background")

# SpliceTransformer

[SpliceTransformer](https://github.com/ShenLab-Genomics/SpliceTransformer) (SpTransformer) is a deep learning model from the Shen Lab that predicts [pre-mRNA splice sites](https://en.wikipedia.org/wiki/RNA_splicing) directly from genomic sequence with tissue-specific resolution. For each position in a target sequence it reports the probability of an acceptor (3' splice site) or donor (5' splice site) along with the predicted usage of that site across fifteen human tissues. This toolkit runs SpliceTransformer through two registered tools: a prediction tool returning per-position probabilities for the three splice-site classes and fifteen tissues, and a gradient tool adding their derivatives.

SpliceTransformer ([You et al., 2024](https://doi.org/10.1038/s41467-024-53088-6)) is a deep learning framework that predicts tissue-specific splicing from genomic sequence. The architecture combines convolutional encoders in the style of [SpliceAI](https://doi.org/10.1016/j.cell.2018.12.015) with a [Sinkhorn transformer](https://en.wikipedia.org/wiki/Transformer_(deep_learning_architecture)) attention module, which lets the model capture the long-range sequence interactions that influence splice site selection. The published work reports that SpliceTransformer outperforms all previous methods on splicing prediction. Applied to roughly 1.3 million variants in the [ClinVar](https://en.wikipedia.org/wiki/ClinVar) database, it attributes 60 percent of intronic and synonymous pathogenic mutations to splicing alterations, and it connects tissue-specific splicing changes to human disease in validations spanning brain disease cohorts and a diabetic nephropathy dataset.

The model evaluates a 1,000 nucleotide target region flanked by 4,000 nucleotides of genomic context on each side, and it returns a prediction for every position in the target region. The first three output channels form a softmax over the splice-site class of the position, namely neither site, acceptor, or donor. The remaining fifteen channels report the predicted usage of the position as a splice site in each of fifteen human tissues, derived from [Genotype-Tissue Expression (GTEx)](https://www.gtexportal.org/) data. The tissue channels are produced by an independent sigmoid for each tissue, so a position can be a confident splice site overall while being used in only a subset of tissues.

## Available tools

In [2]:
display_available_tools("splice_transformer")

- **`run_splice_transformer_gradient()`** — Compute SpliceTransformer splice-channel gradients for a relaxed target region
- **`run_splice_transformer()`** — Tissue-specific splicing prediction using SpliceTransformer

### `run_splice_transformer`

Given a target DNA sequence flanked by left and right genomic context, `run_splice_transformer` uses a transformer architecture trained on RNA-seq data from 15 human tissues to predict per-position splice site probabilities. The output tensor has 18 channels. Three splice type channels (neither, acceptor, donor) sum to 1.0 at every position. They are followed by 15 tissue-specific usage channels covering adipose, blood, brain, heart, liver, and other tissues. This structure makes it straightforward to identify canonical splice sites and to compare their predicted usage across tissues.

In [3]:
import numpy as np

from proto_tools import (
    SPLICE_TISSUE_CHANNEL_INDEX,
    SpliceTransformerType,
    SpliceTransformerConfig,
    SpliceTransformerInput,
    run_splice_transformer,
)

In [4]:
# Display input docs
display_api_reference("splice_transformer", "input", "run_splice_transformer")

from pathlib import Path
from Bio import SeqIO

# Load human beta-globin (HBB) genomic sequence (NG_059281.1)
# HBB is a small, well-characterized gene with 3 exons and 2 introns
fasta_path = Path("./hbb_genomic.fasta")
record = SeqIO.read(fasta_path, "fasta")
full_seq = str(record.seq)
print(f"Gene: HBB (Homo sapiens hemoglobin subunit beta)")
print(f"Accession: {record.id}")
print(f"Total length: {len(full_seq):,}bp")

# Targets must be exactly 1000 bp; left/right contexts exactly 4000 bp.
target_start = 4700
target_end = 5700
target_seq = full_seq[target_start:target_end]
left_ctx = full_seq[target_start - 4000:target_start]
right_ctx = full_seq[target_end:target_end + 4000]

print(f"\nTarget: positions {target_start}-{target_end} ({len(target_seq)}bp)")
print(f"Context: {len(left_ctx)}bp left, {len(right_ctx)}bp right")

inputs = SpliceTransformerInput(
    target_seqs=[target_seq],
    left_contexts=[left_ctx],
    right_contexts=[right_ctx],
)

**Input** — `SpliceTransformerInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>target_seqs</code> | <code>list[str]</code> | required | Target region(s) for splice prediction; each must be exactly 1000 bp. |
| <code>left_contexts</code> | <code>list[str]</code> | required | Left (5') flanking context; each exactly 4000 bp, one per target sequence. |
| <code>right_contexts</code> | <code>list[str]</code> | required | Right (3') flanking context; each exactly 4000 bp, one per target sequence. |

Gene: HBB (Homo sapiens hemoglobin subunit beta)
Accession: NG_059281.1
Total length: 10,106bp

Target: positions 4700-5700 (1000bp)
Context: 4000bp left, 4000bp right


In [5]:
# Display config docs
display_api_reference("splice_transformer", "config", "run_splice_transformer")

config = SpliceTransformerConfig(
    device="cuda",  # Change to "cpu" if no GPU available
    verbose=True,
)

**Config** — `SpliceTransformerConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>device</code> | <code>str</code> | <code>'cuda'</code> | Device to run the model on |

In [6]:
# Run the prediction
result = run_splice_transformer(inputs, config)

Running inference.py (one-shot) with device=cuda:0


Running run_splice_transformer [00:00]

In [7]:
# Display output docs
display_api_reference("splice_transformer", "output", "run_splice_transformer")

pred = np.array(result.prediction)
print(f"Prediction shape: {pred.shape}")
print(f"  batch={pred.shape[0]}, target_length={pred.shape[1]}, channels={pred.shape[2]}")

**Output** — `SpliceTransformerOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>prediction</code> | <code>list[list[list[float]]]</code> | required | Probabilities for 3 splice-site classes + 15 tissues (shape [batch, target_length, 18]) |

Prediction shape: (1, 1000, 18)
  batch=1, target_length=1000, channels=18


## Interpreting Predictions

The prediction tensor has 18 channels. Channels 0 through 2 correspond to splice type probabilities for "neither" (not a splice site), "acceptor", and "donor". Channels 3 through 17 provide tissue-specific splice usage predictions for 15 human tissues. The splice type channels sum to 1.0 at each position, representing a probability distribution over the three categories.

In [8]:
pred = np.array(result.prediction[0])  # First (only) sequence

# Splice type probabilities
neither_probs = pred[:, SpliceTransformerType.NEITHER.value]
acceptor_probs = pred[:, SpliceTransformerType.ACCEPTOR.value]
donor_probs = pred[:, SpliceTransformerType.DONOR.value]

print(f"Max acceptor probability: {acceptor_probs.max():.4f} at position {acceptor_probs.argmax()}")
print(f"Max donor probability: {donor_probs.max():.4f} at position {donor_probs.argmax()}")
print(f"Mean 'neither' probability: {neither_probs.mean():.4f}")

Max acceptor probability: 0.9914 at position 572
Max donor probability: 0.9857 at position 441
Mean 'neither' probability: 0.9960


## Finding High-Confidence Splice Sites

A threshold on acceptor and donor probabilities identifies positions where the model is confident a splice site exists. A threshold of 0.5 typically captures the strongest splice sites while filtering noise.

In [9]:
threshold = 0.5

acceptor_sites = np.where(acceptor_probs > threshold)[0]
donor_sites = np.where(donor_probs > threshold)[0]

print(f"High-confidence acceptor sites (>{threshold}): {len(acceptor_sites)}")
if len(acceptor_sites) > 0:
    print(f"  Positions: {acceptor_sites[:10]}..." if len(acceptor_sites) > 10 else f"  Positions: {acceptor_sites}")

print(f"High-confidence donor sites (>{threshold}): {len(donor_sites)}")
if len(donor_sites) > 0:
    print(f"  Positions: {donor_sites[:10]}..." if len(donor_sites) > 10 else f"  Positions: {donor_sites}")

High-confidence acceptor sites (>0.5): 2
  Positions: [332 572]
High-confidence donor sites (>0.5): 2
  Positions: [441 794]


## Tissue-Specific Analysis

Channels 3 through 17 provide splice usage predictions for 15 human tissues including brain, heart, liver, and muscle. Comparing these channels reveals tissue-specific alternative splicing, where the same splice site is used at different rates in different tissues. This is biologically important because alternative splicing is a major mechanism for generating protein diversity from a single gene.

In [10]:
pred = np.array(result.prediction[0])
for tissue_name, channel_idx in SPLICE_TISSUE_CHANNEL_INDEX.items():
    if channel_idx is None:  # AVERAGE is handled separately via channel slice 3:
        continue
    tissue_pred = pred[:, channel_idx]
    print(f"{tissue_name:20s}: mean={tissue_pred.mean():.4f}, max={tissue_pred.max():.4f}")

ADIPOSE_TISSUE      : mean=0.0026, max=0.6767
BLOOD               : mean=0.0014, max=0.3695
BLOOD_VESSEL        : mean=0.0024, max=0.6270
BRAIN               : mean=0.0026, max=0.6755
COLON               : mean=0.0028, max=0.7136
HEART               : mean=0.0023, max=0.6058
KIDNEY              : mean=0.0029, max=0.7335
LIVER               : mean=0.0021, max=0.5648
LUNG                : mean=0.0030, max=0.7633
MUSCLE              : mean=0.0020, max=0.5354
NERVE               : mean=0.0028, max=0.7103
SMALL_INTESTINE     : mean=0.0030, max=0.7714
SKIN                : mean=0.0026, max=0.6632
SPLEEN              : mean=0.0025, max=0.6588
STOMACH             : mean=0.0027, max=0.7100


### `run_splice_transformer_gradient`

The gradient tool turns these predictions into differentiable quantities. Each term names an output channel — a splice class (`NEITHER`/`ACCEPTOR`/`DONOR`) or one of the fifteen tissues — and a span of target positions, and comes back with that channel's mean probability and its raw gradient. A direction and a weight are goals the caller applies afterwards.

Only the **target region** is relaxed. A candidate bundles the two fixed 4,000 bp flanking contexts with a `(1000, 4)` relaxed target matrix, so the gradient covers the target positions alone while both flanks still shape the prediction. That matches how the model is used. The flanks are genomic context you are not designing.

Here the term covers the strongest acceptor site found above.

In [11]:
from proto_tools import (
    SpliceTransformerGradientCandidate, SpliceTransformerGradientConfig,
    SpliceTransformerGradientInput, SpliceTransformerProbabilityTerm,
    run_splice_transformer_gradient,
)

display_api_reference("splice-transformer-gradient", "input", "run_splice_transformer_gradient")

# Relax the same target region: a sharpened one-hot of the real sequence, so the starting
# distribution favours the wild-type base at each position without saturating.
grad_sharpness = 2.0
grad_logits = [
    [grad_sharpness if base == observed else 0.0 for base in "ACGT"]
    for observed in target_seq.upper()
]
grad_candidate = SpliceTransformerGradientCandidate(
    left_context=left_ctx,
    right_context=right_ctx,
    logits=grad_logits,
)
print(f"contexts: {len(left_ctx)} + {len(right_ctx)} bp | relaxed target: {len(grad_logits)} bp")


**Input** — `SpliceTransformerGradientInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>temperature</code> | <code>float &#124; None</code> | <code>1.0</code> | Softmax temperature for softmax(logits / T); None consumes probabilities directly. |
| <code>inputs</code> | <code>list[SpliceTransformerGradientCandidate]</code> | required | One relaxed target region with its flanking contexts per candidate |

contexts: 4000 + 4000 bp | relaxed target: 1000 bp


In [12]:
display_api_reference("splice-transformer-gradient", "config", "run_splice_transformer_gradient")

# Target an 11 bp window centred on the strongest acceptor position. Term positions are
# 1-indexed and relative to the target region, the same frame `acceptor_probs` indexes.
grad_peak = int(acceptor_probs.argmax())
grad_config = SpliceTransformerGradientConfig(
    gradient_terms=[
        SpliceTransformerProbabilityTerm(
            channel="ACCEPTOR",
            position_start=max(1, grad_peak + 1 - 5),
            position_end=min(len(grad_logits), grad_peak + 1 + 5),
        )
    ],
)
print(f"strongest acceptor at target position {grad_peak} (p={acceptor_probs[grad_peak]:.4f})")

grad_output = run_splice_transformer_gradient(
    SpliceTransformerGradientInput(inputs=grad_candidate), grad_config
)


**Config** — `SpliceTransformerGradientConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>device</code> | <code>str</code> | <code>'cuda'</code> | Device to run the model on |
| <code>soft</code> | <code>float</code> | <code>1.0</code> | Blend raw input (0) toward softmax probabilities (1); 1.0 recommended for almost all uses. |
| <code>hard</code> | <code>float</code> | <code>0.0</code> | Straight-through coefficient; 1 forwards a hard one-hot to the model. |
| <code>gradient_terms</code> | <code>list[SpliceTransformerProbabilityTerm]</code> | <code>[SpliceTransformerProbabilityTerm(id='channel_probability', channel='ACCEPTOR', position_start=None, position_end=None)]</code> | Channel-and-position quantities to differentiate, by name or as instances |
| <code>compute_gradient</code> | <code>bool</code> | <code>True</code> | Run backward pass and return gradient; set False for forward-only scoring |

strongest acceptor at target position 572 (p=0.9914)


Running run_splice_transformer_gradient [00:00]

In [13]:
display_api_reference("splice-transformer-gradient", "output", "run_splice_transformer_gradient")

grad_result = grad_output.results[0]
grad_term = grad_result.terms[0]
print(f"Target length:      {grad_result.metrics['target_length']}")
print(f"Gradient shape:     ({len(grad_term.gradient)}, {len(grad_term.gradient[0])})")
print(f"  {grad_term.term.channel} positions "
      f"{grad_term.term.position_start}-{grad_term.term.position_end}: "
      f"mean probability {grad_term.term_value:.4f}")

# The derivative at the acceptor base itself, over A, C, G, T.
print(f"Gradient at the acceptor base: {[round(v, 6) for v in grad_term.gradient[grad_peak]]}")


**Output** — `SpliceTransformerGradientOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>results</code> | <code>list[SpliceTransformerGradientResult]</code> | required | One result per input candidate, in input order, each carrying its channel probabilities. |
| <code>vocab</code> | <code>list[str]</code> | required | Symbols defining the column ordering for both the input logits and the returned gradient. |

**`SpliceTransformerGradientResult`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>terms</code> | <code>list[SpliceTransformerTermGradient]</code> | <code>[]</code> | One gradient per requested channel-and-position quantity, in the order requested. |
| <code>metrics</code> | <code>dict[str, Any]</code> | <code>{}</code> | Metrics reported for this item alongside its objective value. |

**`SpliceTransformerTermGradient`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>term</code> | <code>SpliceTransformerProbabilityTerm</code> | required | Channel-and-position quantity answered. |
| <code>term_value</code> | <code>float</code> | required | Raw scalar the model produced for this quantity. |
| <code>gradient</code> | <code>list[list[float]] &#124; None</code> | <code>None</code> | Raw derivative of term_value w.r.t. the input; matches its shape and column order. |
| <code>grad_l2_norm</code> | <code>float &#124; None</code> | <code>None</code> | Euclidean norm of the flattened gradient; computed automatically when a gradient is present. |

**`SpliceTransformerProbabilityTerm`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>id</code> | <code>Literal['channel_probability']</code> | <code>'channel_probability'</code> | Identifies this quantity as a SpliceTransformer mean channel probability. |
| <code>channel</code> | <code>Literal['NEITHER', 'ACCEPTOR', 'DONOR', 'ADIPOSE_TISSUE', 'BLOOD', 'BLOOD_VESSEL', 'BRAIN', 'COLON', 'HEART', 'KIDNEY', 'LIVER', 'LUNG', 'MUSCLE', 'NERVE', 'SMALL_INTESTINE', 'SKIN', 'SPLEEN', 'STOMACH']</code> | <code>'ACCEPTOR'</code> | Output channel to optimize: a splice class or a tissue-specific channel |
| <code>position_start</code> | <code>int &#124; None</code> | <code>None</code> | First target position to average, 1-indexed inclusive; None starts at the first |
| <code>position_end</code> | <code>int &#124; None</code> | <code>None</code> | Last target position to average, 1-indexed inclusive; None ends at the last |

**`GradientMetadata`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>description</code> | <code>str</code> | <code>''</code> | What this scalar quantity measures. |
| <code>units</code> | <code>str &#124; None</code> | <code>None</code> | Physical units of the value, where it has them. |
| <code>suggested_optimization_direction</code> | <code>Literal['higher', 'lower', 'in-range', 'context-dependent']</code> | <code>'context-dependent'</code> | Advisory direction for this quantity; does not affect the sign of the gradient. |
| <code>vocab</code> | <code>list[str]</code> | required | Column identity and order for this gradient. |
| <code>coordinate_space</code> | <code>Literal['logits', 'probabilities']</code> | required | Whether the derivative is w.r.t. logits or probabilities. |
| <code>length</code> | <code>int</code> | required | Number of differentiated rows. |
| <code>positions</code> | <code>list[int] &#124; None</code> | <code>None</code> | 1-indexed input position per row; None means contiguous 1..length. |
| <code>input_fingerprint</code> | <code>str</code> | required | Digest of the differentiated matrix, identifying the point the gradient was taken at. |
| <code>hard</code> | <code>float</code> | <code>0.0</code> | Straight-through coefficient in force. |
| <code>soft</code> | <code>float</code> | <code>1.0</code> | Relaxation blend in force. |

**Metrics** — `SpliceTransformerGradientSampleMetrics` (one set per `results` item)

| Metric | Type | Range | Unit | Description |
|--------|------|-------|------|-------------|
| <code>target_length</code> **(primary)** | <code>int</code> | <code>&gt;= 1</code> |  | Number of scored target positions for this candidate. |

Target length:      1000
Gradient shape:     (1000, 4)
  ACCEPTOR positions 568-578: mean probability 0.0850
Gradient at the acceptor base: [0.000106, 0.000156, -0.000418, 0.000156]


## Export Results

Predictions can be exported to NumPy format for downstream analysis, including integration with variant effect prediction pipelines or visualization in genome browsers.

In [14]:
result.export("splice_transformer_predictions", export_path="./example_output", file_format="npy")
print("Predictions exported to ./example_output/splice_transformer_predictions.npy")

Predictions exported to ./example_output/splice_transformer_predictions.npy
